# 🇱🇰 Sri Lanka Road Network — Average Junction-to-Junction Distance Experiment (A → A)

This notebook is a **separate experiment notebook**. It does not refetch OSM data — it imports
the CSV exports produced by `sri_lanka_roads_interactive.ipynb` (Section 9: *Export Full Network
for the Junction-to-Junction Distance Experiment*) and runs the full A→A graph-distance analysis
described in the experiment specification:

1. Build an undirected, weighted graph from the full road network (nodes = junctions/intersections,
   edges = road segments weighted by physical length in meters).
2. For every junction **A**, run Dijkstra's algorithm to every other reachable junction.
3. Compute network-wide, shortest-path, and per-junction statistics.
4. Visualize the results (histogram, box plot, centrality heat map, cumulative distribution).
5. Export a per-junction CSV and a summary report.

**Required input files** (produced by the main notebook's export section, expected in `./data/`
or wherever you point `DATA_DIR` below):

- `sri_lanka_full_network_nodes.csv` — every node in the fetched OSM graph (`node_id`, `latitude`, `longitude`, `street_count`)
- `sri_lanka_full_network_edges.csv` — every road segment (`u`, `v`, `length_m`, ...)
- `sri_lanka_all_junctions_combined.csv` — the AA+B junction nodes that define the **A** set for this experiment


In [ ]:
# ── 0. Dependencies & Configuration ──────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

import warnings
warnings.filterwarnings('ignore')

# Folder containing the 3 CSVs exported from the main notebook
DATA_DIR = "data"          # <- change this if your CSVs live elsewhere (e.g. "outputs")
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NODES_PATH     = os.path.join(DATA_DIR, "sri_lanka_full_network_nodes.csv")
EDGES_PATH     = os.path.join(DATA_DIR, "sri_lanka_full_network_edges.csv")
JUNCTIONS_PATH = os.path.join(DATA_DIR, "sri_lanka_all_junctions_combined.csv")

for p in [NODES_PATH, EDGES_PATH, JUNCTIONS_PATH]:
    status = "✅ found" if os.path.exists(p) else "❌ MISSING"
    print(f"{status:12s} {p}")

print()
print("If any file is missing: run the main notebook's Section 9 export cell,")
print(f"then copy its outputs/ CSVs into '{DATA_DIR}/' (or update DATA_DIR above).")


In [ ]:
# ── 1. Load exported data ─────────────────────────────────────────────────
df_nodes     = pd.read_csv(NODES_PATH)
df_edges     = pd.read_csv(EDGES_PATH)
df_junctions = pd.read_csv(JUNCTIONS_PATH)

print(f"Full network nodes : {len(df_nodes):,}")
print(f"Full network edges : {len(df_edges):,}")
print(f"A-junctions (AA+B) : {len(df_junctions):,}")

df_nodes.head()


---
## 2. Build the Graph

Undirected, weighted graph. Nodes are road junctions/intersections; edges are road segments
weighted by physical length in meters (`length_m`). OSMnx stores each two-way road as two
directed edges (u→v and v→u) — building an undirected `networkx.Graph` and keeping the
**minimum** length seen for any (u, v) pair de-duplicates this automatically, and also collapses
any parallel multi-edges down to the shortest physical connection between the two junctions.


In [ ]:
# ── 2. Build undirected weighted graph ───────────────────────────────────────
G = nx.Graph()

# Node attributes: lat/lon + any available metadata
node_attrs = df_nodes.set_index('node_id').to_dict(orient='index')
for node_id, attrs in node_attrs.items():
    G.add_node(node_id, **attrs)

# Edges: keep the minimum length_m for any duplicate / parallel (u, v) pair
edge_min = (
    df_edges
    .dropna(subset=['u', 'v', 'length_m'])
    .groupby(['u', 'v'], as_index=False)['length_m']
    .min()
)

n_edges_added = 0
n_edges_skipped = 0
for _, row in edge_min.iterrows():
    u, v, w = row['u'], row['v'], row['length_m']
    if u == v or w <= 0:
        n_edges_skipped += 1
        continue
    if G.has_edge(u, v):
        # keep the shorter of the two directions if both appeared
        G[u][v]['weight'] = min(G[u][v]['weight'], w)
    else:
        G.add_edge(u, v, weight=w)
        n_edges_added += 1

print(f"Graph built: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"  edges skipped (self-loops / zero-length): {n_edges_skipped:,}")

# Restrict the "A" junction set to nodes that actually exist in the graph
junction_ids = [j for j in df_junctions['node_id'].unique() if j in G]
missing_junctions = len(df_junctions['node_id'].unique()) - len(junction_ids)
print(f"\nA-junctions present in graph: {len(junction_ids):,}"
      f"  ({missing_junctions} junction IDs from the CSV were not found in the graph and were dropped)")


---
## 3. Network-Wide Statistics


In [ ]:
# ── 3. Network-wide statistics ────────────────────────────────────────────
n_nodes = G.number_of_nodes()
n_edges = G.number_of_edges()
components = list(nx.connected_components(G))
n_components = len(components)
largest_cc = max(components, key=len)

density = nx.density(G)
avg_degree = (2 * n_edges) / n_nodes if n_nodes else 0.0

network_stats = {
    "Number of junctions (nodes)":    n_nodes,
    "Number of road segments (edges)": n_edges,
    "Number of connected components":  n_components,
    "Largest component size":          len(largest_cc),
    "Graph density":                   density,
    "Average node degree":             avg_degree,
}

print("── Network-wide statistics ──")
for k, v in network_stats.items():
    print(f"  {k:35s}: {v:,.6f}" if isinstance(v, float) else f"  {k:35s}: {v:,}")


---
## 4. The Experiment — Dijkstra from Every Junction A → A

For every junction **A** in the combined junction set, compute the shortest path (by physical
distance) to every other reachable junction, recording both the shortest distance and the hop
count. Complexity is `O(V · (E log V))` via repeated single-source Dijkstra — Floyd–Warshall is
not used since the graph has thousands of nodes.

Routing happens over the **full** road graph (so paths can pass through non-junction pass-through
nodes), but distances are only recorded between pairs of **junctions** (the A set), per the
specification.


In [ ]:
# ── 4. Run Dijkstra from every junction (A -> A) ─────────────────────────────
junction_set = set(junction_ids)

pair_records = []          # one row per ordered (A, A') reachable pair
per_junction_records = []  # one row per junction A

for i, source in enumerate(junction_ids):
    # Single-source Dijkstra over the full graph
    dist_map, path_map = nx.single_source_dijkstra(G, source, weight='weight')

    reachable_dists = []
    reachable_hops  = []
    for target in junction_ids:
        if target == source:
            continue
        if target in dist_map:
            d = dist_map[target]
            hops = len(path_map[target]) - 1
            pair_records.append((source, target, d, hops))
            reachable_dists.append(d)
            reachable_hops.append(hops)

    n_reachable   = len(reachable_dists)
    n_unreachable = (len(junction_ids) - 1) - n_reachable

    per_junction_records.append({
        "junction_id":       source,
        "latitude":          G.nodes[source].get("latitude", np.nan),
        "longitude":         G.nodes[source].get("longitude", np.nan),
        "reachable_nodes":   n_reachable,
        "unreachable_nodes": n_unreachable,
        "average_distance":  float(np.mean(reachable_dists)) if reachable_dists else np.nan,
        "maximum_distance":  float(np.max(reachable_dists))  if reachable_dists else np.nan,
        "average_hops":      float(np.mean(reachable_hops))  if reachable_hops  else np.nan,
    })

    if (i + 1) % 200 == 0 or (i + 1) == len(junction_ids):
        print(f"  processed {i + 1:,} / {len(junction_ids):,} source junctions...")

df_pairs = pd.DataFrame(pair_records, columns=["source", "target", "distance_m", "hops"])
df_per_junction = pd.DataFrame(per_junction_records)

print(f"\n✅ Computed shortest paths for {len(junction_ids):,} source junctions")
print(f"   Total reachable ordered pairs: {len(df_pairs):,}")


---
## 5. Shortest-Path Statistics

Self-distances are excluded by construction (the loop above skips `target == source`).
Unreachable pairs are reported separately rather than included in the distance statistics.


In [ ]:
# ── 5. Shortest-path statistics ───────────────────────────────────────────
distances_m = df_pairs["distance_m"].values
hops = df_pairs["hops"].values

n_possible_ordered_pairs = len(junction_ids) * (len(junction_ids) - 1)
n_reachable_pairs = len(df_pairs)
n_unreachable_pairs = n_possible_ordered_pairs - n_reachable_pairs

path_stats = {
    "Mean shortest-path distance (m)":   float(np.mean(distances_m)),
    "Median shortest-path distance (m)": float(np.median(distances_m)),
    "Minimum shortest-path distance (m)": float(np.min(distances_m)),
    "Maximum shortest-path distance (m)": float(np.max(distances_m)),
    "Standard deviation (m)":            float(np.std(distances_m)),
    "95th percentile distance (m)":      float(np.percentile(distances_m, 95)),
    "Mean hop count":                    float(np.mean(hops)),
    "Graph diameter (max shortest path, m)": float(np.max(distances_m)),
}

print("── Shortest-path statistics (junction A -> A, self-distances excluded) ──")
for k, v in path_stats.items():
    print(f"  {k:42s}: {v:,.2f}")

print()
print(f"  Reachable ordered pairs   : {n_reachable_pairs:,} / {n_possible_ordered_pairs:,}"
      f"  ({100 * n_reachable_pairs / n_possible_ordered_pairs:.2f}%)")
print(f"  Unreachable ordered pairs : {n_unreachable_pairs:,}"
      f"  ({100 * n_unreachable_pairs / n_possible_ordered_pairs:.2f}%)")


---
## 6. Per-Junction Statistics (Centrality Proxy)

`average_distance` and `maximum_distance` per junction were already computed during the Dijkstra
pass in Section 4. Lower average distance ⇒ more central; higher ⇒ more remote/isolated.


In [ ]:
# ── 6. Per-junction statistics preview ───────────────────────────────────────
df_per_junction_sorted = df_per_junction.sort_values("average_distance")

print("Most central junctions (lowest average distance to all other junctions):")
display(df_per_junction_sorted.head(10))

print("\nMost remote/isolated junctions (highest average distance):")
display(df_per_junction_sorted.dropna(subset=["average_distance"]).tail(10))


---
## 7. Visualizations

### 7a. Histogram — Distribution of Shortest-Path Distances


In [ ]:
# ── 7a. Histogram ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(distances_m / 1000, bins=60, color="#2980b9", edgecolor="white", alpha=0.85)
ax.axvline(np.mean(distances_m) / 1000, color="#e74c3c", linestyle="--", linewidth=2,
           label=f"Mean = {np.mean(distances_m)/1000:.1f} km")
ax.axvline(np.median(distances_m) / 1000, color="#27ae60", linestyle="--", linewidth=2,
           label=f"Median = {np.median(distances_m)/1000:.1f} km")
ax.set_xlabel("Shortest-path distance (km)")
ax.set_ylabel("Number of junction pairs")
ax.set_title("Distribution of Junction-to-Junction Shortest-Path Distances", fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart_distance_histogram.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved {OUTPUT_DIR}/chart_distance_histogram.png")


### 7b. Box Plot — Shortest-Path Distance Distribution

In [ ]:
# ── 7b. Box plot ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
bp = ax.boxplot(distances_m / 1000, vert=True, patch_artist=True, widths=0.4,
                 showfliers=True, flierprops=dict(marker='o', markersize=2, alpha=0.3))
for box in bp['boxes']:
    box.set_facecolor("#8e44ad")
    box.set_alpha(0.7)
ax.set_ylabel("Shortest-path distance (km)")
ax.set_xticks([1])
ax.set_xticklabels(["All junction pairs"])
ax.set_title("Shortest-Path Distance — Box Plot", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart_distance_boxplot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved {OUTPUT_DIR}/chart_distance_boxplot.png")


### 7c. Heat Map — Junction Centrality

Junctions colored by their average distance to all other junctions:
🟢 green = very central, 🟡 yellow = average, 🔴 red = isolated.


In [ ]:
# ── 7c. Centrality heat map (static, matplotlib) ─────────────────────────────
plot_df = df_per_junction.dropna(subset=["average_distance", "latitude", "longitude"])

fig, ax = plt.subplots(figsize=(11, 14), facecolor="#0d1117")
ax.set_facecolor("#0d1117")

norm = mcolors.Normalize(vmin=plot_df["average_distance"].min(),
                          vmax=plot_df["average_distance"].max())
cmap = cm.get_cmap("RdYlGn_r")   # green (low/central) -> red (high/isolated)

sc = ax.scatter(plot_df["longitude"], plot_df["latitude"],
                 c=plot_df["average_distance"], cmap=cmap, norm=norm,
                 s=18, alpha=0.9, edgecolors="none")

cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label("Average distance to all other junctions (m)", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="white")

ax.set_title("Junction Centrality Heat Map\n(green = central, red = isolated)",
             color="white", fontweight="bold", fontsize=13)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart_centrality_heatmap.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print(f"✅ Saved {OUTPUT_DIR}/chart_centrality_heatmap.png")


### 7c (interactive). Centrality Heat Map — Folium

An interactive Leaflet version of the same centrality heat map: click any junction to see its
exact average and maximum distance.


In [ ]:
# ── 7c-interactive. Folium centrality map ────────────────────────────────────
import folium
import matplotlib.colors as mcolors

m = folium.Map(
    location=[plot_df["latitude"].mean(), plot_df["longitude"].mean()],
    zoom_start=8, tiles="CartoDB dark_matter", control_scale=True,
)

vmin, vmax = plot_df["average_distance"].min(), plot_df["average_distance"].max()
cmap = cm.get_cmap("RdYlGn_r")

for _, row in plot_df.iterrows():
    frac = (row["average_distance"] - vmin) / (vmax - vmin) if vmax > vmin else 0.5
    color = mcolors.to_hex(cmap(frac))
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4, color=color, weight=1, fill=True, fill_color=color, fill_opacity=0.85,
        popup=(f"<b>Junction {int(row['junction_id'])}</b><br>"
               f"Avg distance: {row['average_distance']/1000:.1f} km<br>"
               f"Max distance: {row['maximum_distance']/1000:.1f} km<br>"
               f"Reachable junctions: {int(row['reachable_nodes'])}"),
    ).add_to(m)

interactive_map_path = f"{OUTPUT_DIR}/junction_centrality_map.html"
m.save(interactive_map_path)
print(f"✅ Interactive centrality map saved -> {interactive_map_path}")
m


### 7d. Cumulative Distribution — Junction Pairs vs. Shortest-Path Distance

In [ ]:
# ── 7d. Cumulative distribution ───────────────────────────────────────────
sorted_d = np.sort(distances_m) / 1000
cum_pct = np.arange(1, len(sorted_d) + 1) / len(sorted_d) * 100

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sorted_d, cum_pct, color="#16a085", linewidth=2)
ax.fill_between(sorted_d, cum_pct, alpha=0.15, color="#16a085")
ax.axhline(95, color="#e74c3c", linestyle="--", linewidth=1,
           label=f"95th percentile = {path_stats['95th percentile distance (m)']/1000:.1f} km")
ax.set_xlabel("Shortest-path distance (km)")
ax.set_ylabel("Cumulative percentage of junction pairs (%)")
ax.set_title("Cumulative Distribution of Junction-to-Junction Distances", fontweight="bold")
ax.legend()
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/chart_cumulative_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Saved {OUTPUT_DIR}/chart_cumulative_distribution.png")


---
## 8. Outputs

### 8a. Per-Junction CSV


In [ ]:
# ── 8a. Per-junction CSV ──────────────────────────────────────────────────
csv_export = df_per_junction[[
    "junction_id", "latitude", "longitude",
    "reachable_nodes", "average_distance", "maximum_distance",
]].rename(columns={"junction_id": "junction_id"})
csv_export = csv_export.round({"latitude": 6, "longitude": 6,
                                "average_distance": 2, "maximum_distance": 2})

per_junction_path = f"{OUTPUT_DIR}/junction_distance_stats.csv"
csv_export.to_csv(per_junction_path, index=False)
print(f"✅ Per-junction CSV -> {per_junction_path}  ({len(csv_export):,} rows)")

# Also save the raw pairwise distance table (useful for later experiments,
# e.g. betweenness centrality / robustness simulations)
pairs_path = f"{OUTPUT_DIR}/junction_pairwise_distances.csv"
df_pairs.to_csv(pairs_path, index=False)
print(f"✅ Pairwise distance table -> {pairs_path}  ({len(df_pairs):,} rows)")


### 8b. Summary Report

In [ ]:
# ── 8b. Summary report ────────────────────────────────────────────────────
print("=" * 65)
print("  SRI LANKA ROAD NETWORK — A→A JUNCTION DISTANCE EXPERIMENT")
print("=" * 65)
print()
print(f"  Nodes (junctions analysed)   : {n_nodes:,}")
print(f"  Edges (road segments, full graph) : {n_edges:,}")
print(f"  Connected components         : {n_components:,}")
print(f"  Average degree               : {avg_degree:.3f}")
print(f"  Graph density                : {density:.6f}")
print()
print(f"  Average shortest-path distance : {path_stats['Mean shortest-path distance (m)']/1000:.2f} km")
print(f"  Median distance                : {path_stats['Median shortest-path distance (m)']/1000:.2f} km")
print(f"  Maximum distance               : {path_stats['Maximum shortest-path distance (m)']/1000:.2f} km")
print(f"  Minimum distance               : {path_stats['Minimum shortest-path distance (m)']/1000:.2f} km")
print(f"  Standard deviation             : {path_stats['Standard deviation (m)']/1000:.2f} km")
print(f"  Graph diameter (max obs.)      : {path_stats['Graph diameter (max shortest path, m)']/1000:.2f} km")
print()
print(f"  Reachable junction pairs       : {n_reachable_pairs:,} / {n_possible_ordered_pairs:,} "
      f"({100*n_reachable_pairs/n_possible_ordered_pairs:.2f}%)")
print(f"  Unreachable junction pairs     : {n_unreachable_pairs:,}")
print()
print("  Most central junction  :",
      f"id={df_per_junction_sorted.iloc[0]['junction_id']:.0f}",
      f"(avg dist {df_per_junction_sorted.iloc[0]['average_distance']/1000:.1f} km)")
print("  Most isolated junction :",
      f"id={df_per_junction_sorted.dropna(subset=['average_distance']).iloc[-1]['junction_id']:.0f}",
      f"(avg dist {df_per_junction_sorted.dropna(subset=['average_distance']).iloc[-1]['average_distance']/1000:.1f} km)")
print("=" * 65)


---
## 9. Research Questions — Answered from the Data Above

1. **How well connected is the network?** See *Section 3* (connected components, density,
   average degree) and the reachable-pairs percentage in *Section 5/8b*.
2. **Average travel distance between two arbitrary junctions?** `Mean shortest-path distance`
   in *Section 5/8b*.
3. **Which junctions are most central / remote?** *Section 6* and the heat maps in *Section 7c*.
4. **Which regions look isolated?** Red clusters on the heat map (*Section 7c*) and the tail of
   the per-junction table sorted by `average_distance` (*Section 6*).
5. **Efficient connectivity or excessive path lengths?** Compare the mean/median distance and the
   shape of the cumulative distribution (*Section 7d*) against the network's geographic extent —
   a long right tail or a mean far above the median suggests inefficiently long detours for some
   junction pairs.

This per-junction CSV and pairwise distance table are saved so they can be reused directly for
follow-up experiments: betweenness centrality, articulation-point detection, or network
robustness/failure simulations.
